# 04 | Supporting work: giving Scout a case history

The first version treated each question separately. That is no use when an investigator says "we use PostgreSQL" *after* describing the original timeout.

I added a case with a **unique ID**, messages, evidence snapshots and separate human notes. Scout can use the earlier problem description in a follow-up without treating its own previous guesses as confirmed facts.

## 1. What I save

- **Case:** one investigation and its ID and status.
- **Messages:** what the person and Scout said, in order, with sources.
- **Notes:** observations, confirmed facts, hypotheses and verified resolutions.

I used **SQLite** because it comes with Python and suits this single-user local prototype. Case history is not uploaded to GitHub.

## 2. Test storage without using a real person's case

This example uses a temporary SQLite file, saves both sides of a conversation and reopens the messages. The temporary file is removed when the cell finishes.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.scout_case_store import (
    initialise_database,
    create_case,
    add_message,
    get_case,
    get_messages,
    add_note,
    get_notes,
    set_case_status,
)
from src.scout_conversation import retrieval_query, case_context_for_model

with TemporaryDirectory() as folder:
    test_database = Path(folder) / "scout_example.db"
    initialise_database(test_database)

    case = create_case("Checkout delays", test_database)
    add_message(case["id"], "user", "Checkout is failing and our API is timing out.", db_path=test_database)
    add_message(case["id"], "assistant", "I'd compare the failure times first.", db_path=test_database)
    add_message(case["id"], "user", "We use PostgreSQL and deployed yesterday.", db_path=test_database)

    add_note(case["id"], "confirmed_fact", "We use PostgreSQL.", test_database)
    add_note(case["id"], "hypothesis", "A shared database dependency might be involved.", test_database)

    print("Case:", get_case(case["id"], test_database)["title"])
    print("Messages:", len(get_messages(case["id"], test_database)))
    print("Notes:", len(get_notes(case["id"], test_database)))
    print("Status:", get_case(case["id"], test_database)["status"])

## 3. Keep the earlier question in a follow-up

A reply like "We use PostgreSQL" needs its earlier context. I combine the original question with new **user-supplied** details and any explicitly confirmed notes. Scout's previous answer remains labelled as a suggestion, not evidence.

This is saved conversation context, **not model retraining**.

In [ ]:
example_messages = [
    {"role": "user", "content": "Checkout is failing and our API is timing out."},
    {"role": "assistant", "content": "It could be a database problem, but that is unconfirmed."},
    {"role": "user", "content": "We use PostgreSQL and deployed yesterday."},
]

example_notes = [
    {"kind": "confirmed_fact", "content": "We use PostgreSQL."},
    {"kind": "hypothesis", "content": "The deployment may have affected a shared dependency."},
]

search_text = retrieval_query(
    example_messages,
    example_notes,
    "We use PostgreSQL and deployed yesterday.",
)
print("Search now includes:")
print(search_text)

print("\nCase context sent to Scout:")
print(case_context_for_model(example_messages, example_notes))

## 4. Prevent unverified case closure

Scout cannot decide that a suggested fix worked. The person has to record a **verified resolution** before the case can be closed. The code below tests that rule.

In [ ]:
with TemporaryDirectory() as folder:
    path = Path(folder) / "resolution_test.db"
    initialise_database(path)
    case = create_case("Test closure", path)

    add_note(case["id"], "hypothesis", "Possibly a capacity issue.", path)
    try:
        set_case_status(case["id"], "resolved", path)
    except ValueError as error:
        print("Blocked as intended:", error)

    add_note(case["id"], "verified_resolution",
             "The tester checked the result in this made-up example.", path)
    set_case_status(case["id"], "resolved", path)
    print("Now resolved:", get_case(case["id"], path)["status"])

## 5. What the tests cover

I tested saving and reopening cases, keeping separate cases separate, retaining evidence, using follow-up context and preventing closure without a human-recorded resolution.

These tests check the **software behaviour**, not whether Llama's diagnosis is right.

## 6. A real UI bug I found

On 10 November, a follow-up after a 504 timeout investigation sometimes returned the screen to **Start here**. The case picker was being rebuilt as the case list changed, which could lose the active selection.

I separated the selected case ID from the sidebar widget state. Later, human-review saves showed a similar reset risk; I kept the case list in a stable order and made a new browser session reopen the most recently updated saved case.

The full case ID and a readable transcript are now accessible from **Full case log**. The tests cover reopening, switching cases and saving a review without overwriting the conversation.

**Limit:** a passing test doesn't guarantee the behaviour on every browser and machine. I still check the live app before presenting.

## 7. What I learnt

Keeping a conversation is not enough: the user must also be able to **find the right case again**, see what was saved and tell facts from assumptions.

The local case log now exposes the full UUID, messages, sources, notes and human-review history. I keep the underlying SQLite file out of Git and test new-case and follow-up paths separately.

## 8. The final case-selection fix

During rehearsal I prepared a fresh demo, yet an older investigation's progress reappeared. The full log showed its original UUID and timestamps. I found the case selector was displaying identical names for different demos. I changed the labels to include the case ID and tied each presenter button to the particular case and guide.

I now check a fresh demo alongside an old demo with three completed checks. After completing the fresh timeline, only the new case should have one completed check; the older case must be untouched. I confirmed the corrected demo worked in the browser. This was a UI state problem, not a failure of the saved evidence.